In [22]:
import shutil
import tempfile
import time
from pathlib import Path

import numpy as np
import xarray as xr
import zarr

# Create a fresh temp directory for all zarr stores in this run
tmpdir = Path(tempfile.mkdtemp(prefix="zarr_explore_"))
print(f"Writing to: {tmpdir}")

n1 = 1000
n2 = 200

fname = str(tmpdir / "baseline.zarr")

t_start = time.perf_counter()
for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    ds = xr.Dataset(
        {
            "y1": (["t"], y1),
        },
        coords={
            "x1": ("t", x1),
            "x2": ("t", x2),
        },
    )
    if i == 0:
        ds.to_zarr(
            fname,
            consolidated=False,
            encoding={
                "x1": {"chunks": (10000,)},
                "x2": {"chunks": (10000,)},
                "y1": {"chunks": (10000,)},
            },
        )
    else:
        ds.to_zarr(fname, append_dim="t", consolidated=False)
duration = time.perf_counter() - t_start

print(f"Baseline (xarray append_dim per iteration): {duration:.3f}s")

Writing to: C:\Users\jenielse\AppData\Local\Temp\zarr_explore_p7nsxzuh
Baseline (xarray append_dim per iteration): 17.126s


## Strategy 1: Direct Zarr API (skip xarray overhead on append)

xarray's `to_zarr` with `append_dim` re-validates metadata each call. Using the zarr API directly to append raw numpy arrays avoids this overhead.

In [23]:
fname1 = str(tmpdir / "direct_zarr.zarr")

t_start = time.perf_counter()

# Create the zarr group with xarray for the first write (sets up metadata/attrs)
x1 = np.arange(n1)
x2 = np.repeat([0.0], n1)
y1 = x1 * x2
ds = xr.Dataset(
    {"y1": (["t"], y1)},
    coords={"x1": ("t", x1), "x2": ("t", x2)},
)
ds.to_zarr(
    fname1,
    consolidated=False,
    encoding={
        "x1": {"chunks": (10000,)},
        "x2": {"chunks": (10000,)},
        "y1": {"chunks": (10000,)},
    },
)

# Now open with zarr directly and append
root = zarr.open(fname1, mode="r+")
for i in range(1, n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    root["x1"].append(x1)
    root["x2"].append(x2)
    root["y1"].append(y1)

duration_direct = time.perf_counter() - t_start
print(f"Strategy 1 (direct zarr append): {duration_direct:.3f}s")

Strategy 1 (direct zarr append): 11.604s


## Strategy 2: Pre-allocate with known final size + region writes

If we know the final size, we can pre-allocate the zarr arrays and write to specific regions. This avoids resize operations entirely.

In [24]:
fname2 = str(tmpdir / "prealloc.zarr")

total_size = n1 * n2

t_start = time.perf_counter()

# Pre-allocate zarr arrays with known final size
store = zarr.open_group(fname2, mode="w")
store.create_array("x1", shape=(total_size,), chunks=(10000,), dtype="float64")
store.create_array("x2", shape=(total_size,), chunks=(10000,), dtype="float64")
store.create_array("y1", shape=(total_size,), chunks=(10000,), dtype="float64")

# Write data in regions
for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    slc = slice(i * n1, (i + 1) * n1)
    store["x1"][slc] = x1
    store["x2"][slc] = x2
    store["y1"][slc] = y1

duration_prealloc = time.perf_counter() - t_start
print(f"Strategy 2 (pre-allocate + region write): {duration_prealloc:.3f}s")

Strategy 2 (pre-allocate + region write): 7.651s


## Strategy 3: Direct zarr with unknown final size (resize as needed)

When final size is unknown, we can still use the direct zarr API but resize in chunk-aligned increments to minimize resize overhead.

In [25]:
fname3 = str(tmpdir / "resize.zarr")

chunk_size = 10000

t_start = time.perf_counter()

# Create zarr arrays with initial zero-size (will grow via append)
store = zarr.open_group(fname3, mode="w")
store.create_array("x1", shape=(0,), chunks=(chunk_size,), dtype="float64")
store.create_array("x2", shape=(0,), chunks=(chunk_size,), dtype="float64")
store.create_array("y1", shape=(0,), chunks=(chunk_size,), dtype="float64")

# Append data - zarr handles the resize internally
for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    store["x1"].append(x1)
    store["x2"].append(x2)
    store["y1"].append(y1)

duration_resize = time.perf_counter() - t_start
print(f"Strategy 3 (zarr resize/append, unknown size): {duration_resize:.3f}s")

Strategy 3 (zarr resize/append, unknown size): 12.832s


## Strategy 4: Batched xarray writes (append every N iterations)

Trade-off: buffer multiple iterations in memory and write less frequently. Data in the buffer is at risk on crash, but disk I/O is reduced.

In [26]:
fname4 = str(tmpdir / "batched.zarr")

batch_size = 10  # Write every 10 iterations (10k points = 1 chunk)

t_start = time.perf_counter()

x1_buf, x2_buf, y1_buf = [], [], []

for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    x1_buf.append(x1)
    x2_buf.append(x2)
    y1_buf.append(y1)

    if (i + 1) % batch_size == 0 or i == n2 - 1:
        ds = xr.Dataset(
            {"y1": (["t"], np.concatenate(y1_buf))},
            coords={
                "x1": ("t", np.concatenate(x1_buf)),
                "x2": ("t", np.concatenate(x2_buf)),
            },
        )
        if i < batch_size:
            ds.to_zarr(
                fname4,
                consolidated=False,
                encoding={
                    "x1": {"chunks": (10000,)},
                    "x2": {"chunks": (10000,)},
                    "y1": {"chunks": (10000,)},
                },
            )
        else:
            ds.to_zarr(fname4, append_dim="t", consolidated=False)
        x1_buf, x2_buf, y1_buf = [], [], []

duration_batched = time.perf_counter() - t_start
print(f"Strategy 4 (batched xarray, batch_size={batch_size}): {duration_batched:.3f}s")

Strategy 4 (batched xarray, batch_size=10): 1.125s


## Strategy 5: xarray `region` writes (pre-allocated, xarray-compatible)

Like Strategy 2 but using xarray's `region` parameter to write slices into a pre-allocated zarr store. Keeps xarray metadata intact.

In [27]:
fname5 = str(tmpdir / "region.zarr")

total_size = n1 * n2

t_start = time.perf_counter()

# Create pre-allocated store with xarray (preserves xarray metadata)
ds_template = xr.Dataset(
    {"y1": (["t"], np.zeros(total_size))},
    coords={
        "x1": ("t", np.zeros(total_size)),
        "x2": ("t", np.zeros(total_size)),
    },
)
ds_template.to_zarr(
    fname5,
    consolidated=False,
    compute=False,
    encoding={
        "x1": {"chunks": (10000,)},
        "x2": {"chunks": (10000,)},
        "y1": {"chunks": (10000,)},
    },
)

# Write regions
for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    ds = xr.Dataset(
        {"y1": (["t"], y1)},
        coords={"x1": ("t", x1), "x2": ("t", x2)},
    )
    region = {"t": slice(i * n1, (i + 1) * n1)}
    ds.to_zarr(fname5, region=region, consolidated=False)

duration_region = time.perf_counter() - t_start
print(f"Strategy 5 (xarray region writes, pre-allocated): {duration_region:.3f}s")

Strategy 5 (xarray region writes, pre-allocated): 9.055s


### Investigating Strategy 5 overhead

Strategy 5 still creates an `xr.Dataset` and calls `to_zarr` each iteration. Let's break down where time is spent: dataset creation vs. the `to_zarr` region write.

In [28]:
fname5b = str(tmpdir / "region_profile.zarr")

total_size = n1 * n2

# Create pre-allocated store
ds_template = xr.Dataset(
    {"y1": (["t"], np.zeros(total_size))},
    coords={
        "x1": ("t", np.zeros(total_size)),
        "x2": ("t", np.zeros(total_size)),
    },
)
ds_template.to_zarr(
    fname5b,
    consolidated=False,
    compute=False,
    encoding={
        "x1": {"chunks": (10000,)},
        "x2": {"chunks": (10000,)},
        "y1": {"chunks": (10000,)},
    },
)

# Time the components separately
time_data_gen = 0.0
time_ds_create = 0.0
time_to_zarr = 0.0

for i in range(n2):
    t0 = time.perf_counter()
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2
    t1 = time.perf_counter()

    ds = xr.Dataset(
        {"y1": (["t"], y1)},
        coords={"x1": ("t", x1), "x2": ("t", x2)},
    )
    t2 = time.perf_counter()

    region = {"t": slice(i * n1, (i + 1) * n1)}
    ds.to_zarr(fname5b, region=region, consolidated=False)
    t3 = time.perf_counter()

    time_data_gen += t1 - t0
    time_ds_create += t2 - t1
    time_to_zarr += t3 - t2

print(f"Data generation:    {time_data_gen:.3f}s")
print(f"xr.Dataset creation: {time_ds_create:.3f}s")
print(f"ds.to_zarr(region): {time_to_zarr:.3f}s")
print(f"Total:              {time_data_gen + time_ds_create + time_to_zarr:.3f}s")
print(
    f"\nto_zarr is {time_to_zarr / (time_data_gen + time_ds_create + time_to_zarr) * 100:.0f}% of total time"
)

Data generation:    0.012s
xr.Dataset creation: 0.056s
ds.to_zarr(region): 8.424s
Total:              8.492s

to_zarr is 99% of total time


### Strategy 5b: Pre-allocate with xarray, but write regions via direct zarr

This combines the best of both: xarray sets up the metadata-rich store, but subsequent writes go directly to zarr arrays (no per-iteration xarray overhead).

In [29]:
fname5c = str(tmpdir / "region_direct.zarr")

total_size = n1 * n2

t_start = time.perf_counter()

# Create pre-allocated store with xarray (preserves xarray metadata)
ds_template = xr.Dataset(
    {"y1": (["t"], np.zeros(total_size))},
    coords={
        "x1": ("t", np.zeros(total_size)),
        "x2": ("t", np.zeros(total_size)),
    },
)
ds_template.to_zarr(
    fname5c,
    consolidated=False,
    compute=False,
    encoding={
        "x1": {"chunks": (10000,)},
        "x2": {"chunks": (10000,)},
        "y1": {"chunks": (10000,)},
    },
)

# Open with zarr directly for fast region writes
root = zarr.open_group(fname5c, mode="r+")
for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    slc = slice(i * n1, (i + 1) * n1)
    root["x1"][slc] = x1
    root["x2"][slc] = x2
    root["y1"][slc] = y1

duration_region_direct = time.perf_counter() - t_start
print(
    f"Strategy 5b (xarray pre-alloc + direct zarr region write): {duration_region_direct:.3f}s"
)
print(f"Speedup vs Strategy 5: {duration_region / duration_region_direct:.1f}x")

# Verify it's still readable as xarray
ds_check = xr.open_zarr(fname5c)
print(f"Readable as xarray: {ds_check.dims}")

Strategy 5b (xarray pre-alloc + direct zarr region write): 7.183s
Speedup vs Strategy 5: 1.3x
Readable as xarray: FrozenMappingWarningOnValuesAccess({'t': 200000})


C:\Users\jenielse\AppData\Local\Temp\ipykernel_16060\3736618097.py:39: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds_check = xr.open_zarr(fname5c)


### Analysis

The profiling shows `ds.to_zarr(region=...)` accounts for **99% of time** in Strategy 5. Each call takes ~57ms of overhead for just 1000 floats (8KB). This overhead comes from xarray re-opening the store, validating schemas, and checking coordinate alignment on every call.

Key findings from the summary table:
- **Strategy 2** (direct zarr region write): ~2x faster — skips xarray validation overhead
- **Strategy 5** (xarray region): barely faster than baseline — same per-call xarray overhead
- **Strategy 4** (batched): **~13x faster** — the real win is reducing call count from 200 → 20

The fundamental bottleneck is **per-call overhead × number of calls**, not raw I/O. Writing 1000 points per call means 200 calls, each with ~30-60ms overhead regardless of data size.

The optimal approach is: **batch writes to align with chunk boundaries + use direct zarr API**.

### Strategy 6: Best of both — xarray metadata + direct zarr + chunk-aligned batching

Pre-allocate with xarray (for metadata), buffer writes in memory, flush to zarr directly when a full chunk is ready. Crash-safe per chunk.

In [30]:
fname6 = str(tmpdir / "best_combined.zarr")

chunk_size = 10000
total_size = n1 * n2

t_start = time.perf_counter()

# 1. Pre-allocate with xarray (sets up coords, attrs, metadata)
ds_template = xr.Dataset(
    {"y1": (["t"], np.zeros(total_size))},
    coords={
        "x1": ("t", np.zeros(total_size)),
        "x2": ("t", np.zeros(total_size)),
    },
)
ds_template.to_zarr(
    fname6,
    consolidated=False,
    compute=False,
    encoding={
        "x1": {"chunks": (chunk_size,)},
        "x2": {"chunks": (chunk_size,)},
        "y1": {"chunks": (chunk_size,)},
    },
)

# 2. Open with zarr for direct writes, buffer to chunk boundaries
root = zarr.open_group(fname6, mode="r+")
write_offset = 0
x1_buf, x2_buf, y1_buf = [], [], []
buf_len = 0

for i in range(n2):
    x1 = np.arange(n1)
    x2 = np.repeat([float(i)], n1)
    y1 = x1 * x2

    x1_buf.append(x1)
    x2_buf.append(x2)
    y1_buf.append(y1)
    buf_len += n1

    # Flush when buffer fills a chunk
    if buf_len >= chunk_size:
        slc = slice(write_offset, write_offset + buf_len)
        root["x1"][slc] = np.concatenate(x1_buf)
        root["x2"][slc] = np.concatenate(x2_buf)
        root["y1"][slc] = np.concatenate(y1_buf)
        write_offset += buf_len
        x1_buf, x2_buf, y1_buf = [], [], []
        buf_len = 0

# Flush remaining
if buf_len > 0:
    slc = slice(write_offset, write_offset + buf_len)
    root["x1"][slc] = np.concatenate(x1_buf)
    root["x2"][slc] = np.concatenate(x2_buf)
    root["y1"][slc] = np.concatenate(y1_buf)

duration_best = time.perf_counter() - t_start
print(
    f"Strategy 6 (xarray metadata + zarr direct + chunk batching): {duration_best:.3f}s"
)
print(f"Speedup vs baseline: {duration / duration_best:.1f}x")

# Verify still readable as xarray
ds_check = xr.open_zarr(fname6, consolidated=False)
print(f"Readable as xarray: {ds_check.dims}")

Strategy 6 (xarray metadata + zarr direct + chunk batching): 0.635s
Speedup vs baseline: 27.0x
Readable as xarray: FrozenMappingWarningOnValuesAccess({'t': 200000})


## Summary comparison

In [31]:
import pandas as pd

results = pd.DataFrame(
    {
        "Strategy": [
            "Baseline (xarray append_dim each iter)",
            "1: Direct zarr append",
            "2: Pre-allocate + region (zarr)",
            "3: Zarr resize/append (unknown size)",
            "4: Batched xarray (batch=10)",
            "5: xarray region writes (pre-alloc)",
        ],
        "Duration (s)": [
            duration,
            duration_direct,
            duration_prealloc,
            duration_resize,
            duration_batched,
            duration_region,
        ],
        "Crash-safe": [
            "per iteration",
            "per iteration",
            "per iteration",
            "per iteration",
            f"per {batch_size} iters",
            "per iteration",
        ],
        "Needs final size": [
            "No",
            "No",
            "Yes",
            "No",
            "No",
            "Yes",
        ],
    }
)
results["Speedup vs baseline"] = (
    results["Duration (s)"].iloc[0] / results["Duration (s)"]
)
# results.style.format({"Duration (s)": "{:.3f}", "Speedup vs baseline": "{:.1f}x"})
results

,Strategy,Duration (s),Crash-safe,Needs final size,Speedup vs baseline
0,Baseline (xarray append_dim each iter),17.125968,per iteration,No,1.000000
1,1: Direct zarr append,11.604002,per iteration,No,1.475867
2,2: Pre-allocate + region (zarr),7.650814,per iteration,Yes,2.238450
3,3: Zarr resize/append (unknown size),12.831808,per iteration,No,1.334650
4,4: Batched xarray (batch=10),1.124639,per 10 iters,No,15.227971
5,5: xarray region writes (pre-alloc),9.054949,per iteration,Yes,1.891338


In [32]:
# Cleanup entire temp directory
shutil.rmtree(tmpdir, ignore_errors=True)
print(f"Cleaned up: {tmpdir}")

Cleaned up: C:\Users\jenielse\AppData\Local\Temp\zarr_explore_p7nsxzuh
